
# FairWarn-SHS — Notebook 11
## Harmonized Final Model Evaluation

### Why this notebook exists

Earlier experiments used different evaluation protocols:

- the original tabular baselines used repeated stratified cross-validation;
- GraphSAGE/FairWarn-SHS used five fixed train/validation/test splits.

Those experiments remain useful as development evidence, but the **final thesis
comparison must evaluate all main models on the same test students for each seed**.

This notebook therefore evaluates, under identical five-seed splits:

1. Logistic Regression
2. Random Forest
3. Feature-only MLP
4. Standard GraphSAGE
5. FairWarn-SHS with the globally selected residence EO regularization strength

It also saves the common test predictions needed for:

- paired statistical tests;
- precision-recall curves;
- confusion matrices;
- stability plots;
- final tables.

### Fairness contribution

The single model modification remains:

\[
L_{\mathrm{FairWarn}} =
L_{\mathrm{weighted\ CE}} + \lambda L_{\mathrm{EO}}
\]

with the globally selected \(\lambda = 2.0\), chosen previously using validation
results only.


In [ ]:
!pip -q install torch-geometric pandas numpy scikit-learn matplotlib scipy

In [ ]:

from google.colab import files
uploaded = files.upload()

# Upload:
# FairWarn_SHS_Node_Features.csv
# FairWarn_SHS_Edge_List.csv


In [ ]:

import time
import random
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from scipy.stats import wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
    brier_score_loss,
    precision_recall_curve,
    confusion_matrix,
)
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

NODE_FILE = "FairWarn_SHS_Node_Features.csv"
EDGE_FILE = "FairWarn_SHS_Edge_List.csv"

SEEDS = [42, 123, 456, 789, 1010]
FAIRNESS_LAMBDA = 2.0
MAX_EPOCHS = 500
PATIENCE = 40

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:

nodes = pd.read_csv(NODE_FILE)
edges = pd.read_csv(EDGE_FILE)

labelled_mask = (
    nodes["Label_Available"].eq(1) &
    nodes["TARGET_AtRisk"].notna()
).to_numpy()

excluded = {
    "Node_ID", "Roster_Code", "School_Code", "Class_Code",
    "Label_Available", "TARGET_AtRisk"
}

feature_columns = [c for c in nodes.columns if c not in excluded]
y_all = nodes["TARGET_AtRisk"].fillna(-1).astype(int).to_numpy()

residence_text = (
    nodes["Q6_Residence"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .str.lower()
)

residence = np.full(len(nodes), -1, dtype=np.int64)
residence[residence_text.str.contains("board", regex=False)] = 0
residence[residence_text.str.contains("day", regex=False)] = 1

node_map = {node_id: i for i, node_id in enumerate(nodes["Node_ID"])}
pairs = []
for _, row in edges.iterrows():
    s_id, t_id = row["Source_Node_ID"], row["Target_Node_ID"]
    if s_id in node_map and t_id in node_map:
        s, t = node_map[s_id], node_map[t_id]
        pairs.extend([(s, t), (t, s)])

edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()

print("Nodes:", len(nodes))
print("Labelled:", int(labelled_mask.sum()))
print("At-risk:", int((y_all[labelled_mask] == 1).sum()))
print("Not-at-risk:", int((y_all[labelled_mask] == 0).sum()))
print("Undirected edges:", edge_index.shape[1] // 2)


In [ ]:

def common_split(seed):
    labelled_indices = np.where(labelled_mask)[0]
    labelled_y = y_all[labelled_indices]

    train_val, test = train_test_split(
        labelled_indices,
        test_size=0.20,
        stratify=labelled_y,
        random_state=seed
    )

    train_val_y = y_all[train_val]

    train, val = train_test_split(
        train_val,
        test_size=0.1875,
        stratify=train_val_y,
        random_state=seed
    )

    return train, val, test


def make_preprocessor(frame):
    numeric = frame.select_dtypes(include=[np.number]).columns.tolist()
    categorical = [c for c in frame.columns if c not in numeric]

    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])


def metrics(y_true, prob, pred):
    return {
        "AUC_ROC": roc_auc_score(y_true, prob),
        "AUC_PR": average_precision_score(y_true, prob),
        "Precision_AtRisk": precision_score(y_true, pred, zero_division=0),
        "Recall_AtRisk": recall_score(y_true, pred, zero_division=0),
        "F1_AtRisk": f1_score(y_true, pred, zero_division=0),
        "Weighted_F1": f1_score(y_true, pred, average="weighted", zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred),
        "Accuracy": accuracy_score(y_true, pred),
        "Brier_Score": brier_score_loss(y_true, prob),
    }


In [ ]:

class GraphModel(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, 64, aggr="mean")
        self.conv2 = SAGEConv(64, 32, aggr="mean")
        self.classifier = torch.nn.Linear(32, 2)
        self.dropout = 0.35

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


def soft_eo_loss(prob, labels, residence_tensor, train_mask):
    positive = train_mask & labels.eq(1) & residence_tensor.ge(0)
    boarding = positive & residence_tensor.eq(0)
    day = positive & residence_tensor.eq(1)

    if boarding.sum() == 0 or day.sum() == 0:
        return torch.tensor(0.0, device=prob.device)

    return torch.abs(prob[boarding].mean() - prob[day].mean())


In [ ]:

result_rows = []
prediction_rows = []
runtime_rows = []
training_history_rows = []

X_raw_all = nodes[feature_columns].copy()

for seed in SEEDS:
    print("\n=== Seed", seed, "===")
    set_seed(seed)

    train_idx, val_idx, test_idx = common_split(seed)

    # Fit one common preprocessing transform using TRAIN nodes only.
    preprocessor = make_preprocessor(X_raw_all.iloc[train_idx])
    X_train = preprocessor.fit_transform(X_raw_all.iloc[train_idx]).astype(np.float32)
    X_val = preprocessor.transform(X_raw_all.iloc[val_idx]).astype(np.float32)
    X_test = preprocessor.transform(X_raw_all.iloc[test_idx]).astype(np.float32)
    X_all_encoded = preprocessor.transform(X_raw_all).astype(np.float32)

    y_train = y_all[train_idx]
    y_test = y_all[test_idx]

    # ---------------- Tabular baselines ----------------
    tabular_models = {
        "Logistic Regression": LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=seed
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        ),
        "Feature-only MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            alpha=0.0005,
            learning_rate_init=0.001,
            max_iter=500,
            early_stopping=True,
            random_state=seed
        ),
    }

    for model_name, model in tabular_models.items():
        started = time.perf_counter()
        model.fit(X_train, y_train)
        train_seconds = time.perf_counter() - started

        infer_started = time.perf_counter()
        prob = model.predict_proba(X_test)[:, 1]
        infer_seconds = time.perf_counter() - infer_started
        pred = (prob >= 0.5).astype(int)

        row = {
            "Seed": seed,
            "Model": model_name,
            **metrics(y_test, prob, pred)
        }
        result_rows.append(row)
        runtime_rows.append({
            "Seed": seed,
            "Model": model_name,
            "Train_Seconds": train_seconds,
            "Inference_Seconds": infer_seconds
        })

        for idx, truth, pclass, p in zip(test_idx, y_test, pred, prob):
            prediction_rows.append({
                "Seed": seed,
                "Model": model_name,
                "Node_Index": int(idx),
                "Node_ID": nodes.iloc[idx]["Node_ID"],
                "True_Label": int(truth),
                "Predicted_Label": int(pclass),
                "AtRisk_Probability": float(p),
            })

        print(model_name, "AUC-PR", round(row["AUC_PR"], 4))

    # ---------------- Graph inputs ----------------
    graph = Data(
        x=torch.tensor(X_all_encoded, dtype=torch.float32),
        edge_index=edge_index,
        y=torch.tensor(y_all, dtype=torch.long),
        residence=torch.tensor(residence, dtype=torch.long)
    )

    train_mask = torch.zeros(len(nodes), dtype=torch.bool)
    val_mask = torch.zeros(len(nodes), dtype=torch.bool)
    test_mask = torch.zeros(len(nodes), dtype=torch.bool)
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    graph.train_mask = train_mask
    graph.val_mask = val_mask
    graph.test_mask = test_mask
    graph = graph.to(device)

    train_labels = graph.y[graph.train_mask]
    counts = torch.bincount(train_labels, minlength=2).float()
    class_weights = (counts.sum() / (2.0 * counts.clamp_min(1.0))).to(device)

    # Standard GraphSAGE and fixed-lambda FairWarn-SHS
    for model_name, fairness_lambda in [
        ("Standard GraphSAGE", 0.0),
        ("FairWarn-SHS", FAIRNESS_LAMBDA),
    ]:
        set_seed(seed)
        model = GraphModel(graph.num_node_features).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

        best_state = None
        best_val_ap = -np.inf
        best_epoch = 0
        wait = 0

        started = time.perf_counter()

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            optimizer.zero_grad()

            logits = model(graph.x, graph.edge_index)
            prob_all = torch.softmax(logits, dim=1)[:, 1]

            ce = F.cross_entropy(
                logits[graph.train_mask],
                graph.y[graph.train_mask],
                weight=class_weights
            )

            fairness = soft_eo_loss(
                prob_all,
                graph.y,
                graph.residence,
                graph.train_mask
            )

            loss = ce + fairness_lambda * fairness
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                logits = model(graph.x, graph.edge_index)
                prob_all = torch.softmax(logits, dim=1)[:, 1]
                val_true = graph.y[graph.val_mask].cpu().numpy()
                val_prob = prob_all[graph.val_mask].cpu().numpy()
                val_ap = average_precision_score(val_true, val_prob)

            training_history_rows.append({
                "Seed": seed,
                "Model": model_name,
                "Epoch": epoch,
                "Training_Loss": float(loss.item()),
                "Validation_AUC_PR": float(val_ap)
            })

            if val_ap > best_val_ap + 1e-6:
                best_val_ap = val_ap
                best_epoch = epoch
                best_state = deepcopy(model.state_dict())
                wait = 0
            else:
                wait += 1

            if wait >= PATIENCE:
                break

        train_seconds = time.perf_counter() - started

        model.load_state_dict(best_state)
        model.eval()

        infer_started = time.perf_counter()
        with torch.no_grad():
            logits = model(graph.x, graph.edge_index)
            prob_all = torch.softmax(logits, dim=1)[:, 1]
            pred_all = torch.argmax(logits, dim=1)
        infer_seconds = time.perf_counter() - infer_started

        truth = graph.y[graph.test_mask].cpu().numpy()
        prob = prob_all[graph.test_mask].cpu().numpy()
        pred = pred_all[graph.test_mask].cpu().numpy()

        row = {
            "Seed": seed,
            "Model": model_name,
            "Best_Epoch": best_epoch,
            **metrics(truth, prob, pred)
        }
        result_rows.append(row)
        runtime_rows.append({
            "Seed": seed,
            "Model": model_name,
            "Train_Seconds": train_seconds,
            "Inference_Seconds": infer_seconds
        })

        for idx, truth_i, pclass, p in zip(test_idx, truth, pred, prob):
            prediction_rows.append({
                "Seed": seed,
                "Model": model_name,
                "Node_Index": int(idx),
                "Node_ID": nodes.iloc[idx]["Node_ID"],
                "True_Label": int(truth_i),
                "Predicted_Label": int(pclass),
                "AtRisk_Probability": float(p),
            })

        print(model_name, "AUC-PR", round(row["AUC_PR"], 4))

results_df = pd.DataFrame(result_rows)
predictions_df = pd.DataFrame(prediction_rows)
runtime_df = pd.DataFrame(runtime_rows)
training_history_df = pd.DataFrame(training_history_rows)


In [ ]:

metric_columns = [
    "AUC_ROC", "AUC_PR", "Precision_AtRisk", "Recall_AtRisk",
    "F1_AtRisk", "Weighted_F1", "Balanced_Accuracy",
    "Accuracy", "Brier_Score"
]

summary_rows = []

for model_name, group in results_df.groupby("Model"):
    row = {"Model": model_name, "Seeds": group["Seed"].nunique()}

    for metric in metric_columns:
        row[f"{metric}_Mean"] = group[metric].mean()
        row[f"{metric}_SD"] = group[metric].std(ddof=1)

    times = runtime_df[runtime_df["Model"].eq(model_name)]
    row["Train_Seconds_Mean"] = times["Train_Seconds"].mean()
    row["Train_Seconds_SD"] = times["Train_Seconds"].std(ddof=1)
    row["Inference_Seconds_Mean"] = times["Inference_Seconds"].mean()
    row["Inference_Seconds_SD"] = times["Inference_Seconds"].std(ddof=1)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values(
    "AUC_PR_Mean", ascending=False
).reset_index(drop=True)

summary_df


In [ ]:

# Pairwise Wilcoxon tests against Standard GraphSAGE and FairWarn-SHS.
comparison_pairs = [
    ("Standard GraphSAGE", "FairWarn-SHS"),
    ("Logistic Regression", "Standard GraphSAGE"),
    ("Random Forest", "Standard GraphSAGE"),
    ("Feature-only MLP", "Standard GraphSAGE"),
]

test_metrics = ["AUC_PR", "Recall_AtRisk", "F1_AtRisk", "Balanced_Accuracy"]
stats_rows = []

for model_a, model_b in comparison_pairs:
    for metric in test_metrics:
        a = (
            results_df[results_df["Model"].eq(model_a)]
            .sort_values("Seed")[metric].to_numpy()
        )
        b = (
            results_df[results_df["Model"].eq(model_b)]
            .sort_values("Seed")[metric].to_numpy()
        )

        diff = b - a

        try:
            statistic, p_value = wilcoxon(
                b, a, zero_method="wilcox", alternative="two-sided"
            )
        except ValueError:
            statistic, p_value = np.nan, np.nan

        nonzero = diff[diff != 0]
        if len(nonzero) == 0:
            rank_biserial = 0.0
        else:
            ranks = pd.Series(np.abs(nonzero)).rank(method="average").to_numpy()
            pos = ranks[nonzero > 0].sum()
            neg = ranks[nonzero < 0].sum()
            rank_biserial = (pos - neg) / (pos + neg)

        stats_rows.append({
            "Model_A": model_a,
            "Model_B": model_b,
            "Metric": metric,
            "Mean_A": a.mean(),
            "Mean_B": b.mean(),
            "Mean_Difference_B_Minus_A": diff.mean(),
            "Wilcoxon_Statistic": statistic,
            "P_Value_Two_Sided": p_value,
            "Rank_Biserial_Effect_Size": rank_biserial,
            "Pairs": len(diff),
        })

paired_stats_df = pd.DataFrame(stats_rows)
paired_stats_df


## Final thesis figures generated from common test predictions

In [ ]:

# Figure 1: AUC-PR stability across seeds.
plt.figure(figsize=(9, 5))

ordered_models = [
    "Logistic Regression",
    "Random Forest",
    "Feature-only MLP",
    "Standard GraphSAGE",
    "FairWarn-SHS",
]

box_data = [
    results_df[results_df["Model"].eq(model)]["AUC_PR"].to_numpy()
    for model in ordered_models
]

plt.boxplot(box_data, tick_labels=ordered_models, showmeans=True)
plt.ylabel("AUC-PR")
plt.title("Five-seed AUC-PR stability")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("figure_auc_pr_stability.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# Figure 2: Mean precision-recall curves across pooled common test predictions.
plt.figure(figsize=(8, 6))

for model_name in ordered_models:
    subset = predictions_df[predictions_df["Model"].eq(model_name)]
    precision, recall, _ = precision_recall_curve(
        subset["True_Label"],
        subset["AtRisk_Probability"]
    )
    plt.plot(recall, precision, label=model_name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curves on pooled common test predictions")
plt.legend()
plt.tight_layout()
plt.savefig("figure_precision_recall_curves.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# Figure 3: Normalized confusion matrices for Standard GraphSAGE and FairWarn-SHS.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, model_name in zip(
    axes,
    ["Standard GraphSAGE", "FairWarn-SHS"]
):
    subset = predictions_df[predictions_df["Model"].eq(model_name)]
    cm = confusion_matrix(
        subset["True_Label"],
        subset["Predicted_Label"],
        labels=[0, 1],
        normalize="true"
    )

    image = ax.imshow(cm)
    ax.set_title(model_name)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks([0, 1], ["Not at risk", "At risk"])
    ax.set_yticks([0, 1], ["Not at risk", "At risk"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center")

fig.tight_layout()
plt.savefig("figure_normalized_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# Figure 4: Training convergence for the two graph models.
plt.figure(figsize=(9, 5))

mean_history = (
    training_history_df
    .groupby(["Model", "Epoch"])["Validation_AUC_PR"]
    .mean()
    .reset_index()
)

for model_name in ["Standard GraphSAGE", "FairWarn-SHS"]:
    subset = mean_history[mean_history["Model"].eq(model_name)]
    plt.plot(
        subset["Epoch"],
        subset["Validation_AUC_PR"],
        label=model_name
    )

plt.xlabel("Epoch")
plt.ylabel("Mean validation AUC-PR")
plt.title("Graph-model validation convergence")
plt.legend()
plt.tight_layout()
plt.savefig("figure_graph_training_convergence.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# Save final harmonized outputs.
summary_df.to_csv("fairwarn11_harmonized_summary_mean_sd.csv", index=False)
results_df.to_csv("fairwarn11_harmonized_metrics_by_seed.csv", index=False)
predictions_df.to_csv("fairwarn11_harmonized_predictions.csv", index=False)
paired_stats_df.to_csv("fairwarn11_paired_statistical_tests.csv", index=False)
runtime_df.to_csv("fairwarn11_runtime_by_seed.csv", index=False)
training_history_df.to_csv("fairwarn11_graph_training_history.csv", index=False)

from google.colab import files

for filename in [
    "fairwarn11_harmonized_summary_mean_sd.csv",
    "fairwarn11_harmonized_metrics_by_seed.csv",
    "fairwarn11_paired_statistical_tests.csv",
    "fairwarn11_runtime_by_seed.csv",
    "figure_auc_pr_stability.png",
    "figure_precision_recall_curves.png",
    "figure_normalized_confusion_matrices.png",
    "figure_graph_training_convergence.png",
]:
    files.download(filename)
